## Visualization for Data Analysis Plan

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import nibabel as nib
from nilearn import datasets, plotting

### General settings for visualization:

In [ ]:
%config InlineBackend.figure_format = "retina"

OUTDIR = "conceptual_poster_figs"
os.makedirs(OUTDIR, exist_ok=True)

# Unified "Nature Neuroscience-ish" styling
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 0.8,
    "font.family": "DejaVu Sans",   # close enough to Helvetica/Arial; swap later in Illustrator if needed
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "savefig.dpi": 300,
})

# Consistent group colors (muted, print-friendly)
LEFT_COLOR  = "#2b6cb0"
RIGHT_COLOR = "#c53030"

POST_TYPES = ["Pro-Left", "Anti-Right", "Pro-Right", "Anti-Left"]

def save_fig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    svg = os.path.join(OUTDIR, f"{name}.svg")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", svg)

In [ ]:
MNI_TEMPLATE = datasets.load_mni152_template(resolution=2)
_TEMPLATE_DATA = MNI_TEMPLATE.get_fdata()
_TEMPLATE_MASK = _TEMPLATE_DATA > 0
_I, _J, _K = np.indices(_TEMPLATE_DATA.shape)
_XYZ = nib.affines.apply_affine(MNI_TEMPLATE.affine, np.column_stack((_I.ravel(), _J.ravel(), _K.ravel())))
_X = _XYZ[:, 0].reshape(_TEMPLATE_DATA.shape)
_Y = _XYZ[:, 1].reshape(_TEMPLATE_DATA.shape)
_Z = _XYZ[:, 2].reshape(_TEMPLATE_DATA.shape)


def _stat_map_from_blobs(mode="isc", overlay_strength=1.0):
    """Create a deterministic 3D statistical map in MNI space."""
    if mode == "isc":
        centers = [(-46, -16, 22), (-8, -56, 18), (42, -26, 30)]
        sigmas = [(12, 10, 9), (9, 12, 10), (10, 11, 9)]
    elif mode == "pattern":
        centers = [(-34, -62, 10), (40, -52, 20)]
        sigmas = [(11, 12, 9), (12, 10, 10)]
    else:  # rsa
        centers = [(-12, -40, 28), (-44, -14, 24), (30, -20, 14)]
        sigmas = [(10, 10, 9), (11, 9, 10), (9, 10, 8)]

    stat = np.zeros_like(_TEMPLATE_DATA, dtype=float)
    for (mx, my, mz), (sx, sy, sz) in zip(centers, sigmas):
        stat += np.exp(
            -(
                ((_X - mx) ** 2) / (2 * sx ** 2)
                + ((_Y - my) ** 2) / (2 * sy ** 2)
                + ((_Z - mz) ** 2) / (2 * sz ** 2)
            )
        )

    stat *= float(overlay_strength)
    stat[~_TEMPLATE_MASK] = 0.0
    return nib.Nifti1Image(stat, MNI_TEMPLATE.affine)


def draw_brain_like(ax, title=None, overlay_strength=1.0, mode="isc"):
    """Draw a Nilearn template background with a deterministic activation overlay."""
    stat_img = _stat_map_from_blobs(mode=mode, overlay_strength=overlay_strength)
    data = stat_img.get_fdata()
    nz = data[data > 0]
    vmax = float(np.percentile(nz, 99)) if nz.size else 1.0

    plotting.plot_stat_map(
        stat_img,
        bg_img=MNI_TEMPLATE,
        display_mode="z",
        cut_coords=[-24, 0, 24],
        cmap="YlOrRd",
        threshold=vmax * 0.15,
        black_bg=False,
        annotate=False,
        colorbar=False,
        axes=ax,
    )

    if title:
        ax.set_title(title, pad=6, loc="left", fontweight="bold")

    sm = plt.cm.ScalarMappable(cmap="YlOrRd", norm=plt.Normalize(vmin=0, vmax=vmax))
    sm.set_array([])
    return sm



In [ ]:
# ---------- (1) ISC dataframe ----------
# Logic:
# - Each group shows higher ISC for "ingroup-favoring" content
# - Moderate for outgroup derogation
# - Lower for outgroup-favoring
df_isc = pd.DataFrame({
    "group": ["Left"]*4 + ["Right"]*4,
    "post_type": POST_TYPES + POST_TYPES,
    "mean_isc": [
        0.22, 0.18, 0.10, 0.12,   # Left group: higher for Pro-Left + Anti-Right
        0.10, 0.12, 0.22, 0.18    # Right group: higher for Pro-Right + Anti-Left
    ]
})
df_isc


# ---------- (2) Post list + deterministic "embeddings" for RSM/RDM ----------
# Create a posts dataframe (18 per type → 72 posts total)
n_per_type = 18
df_posts = pd.DataFrame({
    "post_id": np.arange(4*n_per_type),
    "post_type": np.repeat(POST_TYPES, n_per_type)
})

# Deterministic semantic coordinates (2D) → makes block structure in similarity
# Each post type sits in its own cluster; within-type posts vary smoothly along an angle
type_centers = {
    "Pro-Left":   np.array([+1.8, +0.7]),
    "Anti-Right": np.array([+1.2, -0.6]),
    "Pro-Right":  np.array([-1.8, -0.7]),
    "Anti-Left":  np.array([-1.2, +0.6]),
}

def smooth_circle_offsets(k):
    # deterministic offsets around a small circle (no randomness)
    angles = np.linspace(0, 2*np.pi, k, endpoint=False)
    return np.c_[0.25*np.cos(angles), 0.20*np.sin(angles)]

offsets = smooth_circle_offsets(n_per_type)

coords = []
for pt in POST_TYPES:
    center = type_centers[pt]
    coords.append(center + offsets)
coords = np.vstack(coords)

df_posts["x"] = coords[:,0]
df_posts["y"] = coords[:,1]

df_posts.head()


# ---------- (3) Behavioral measures per post ----------
# Logic consistent with your behavioral finding:
# higher support → lower perceived extremity (inverse relationship)
#
# We also include a "polarization-weighted" distance component.
#
# Build deterministic "ideology score" for post types:
# Pro-Left (+), Anti-Right (+), Pro-Right (-), Anti-Left (-)
type_ideo = {"Pro-Left": +1.0, "Anti-Right": +0.7, "Pro-Right": -1.0, "Anti-Left": -0.7}
ideo = df_posts["post_type"].map(type_ideo).to_numpy()

# Support: monotonic with ideology score magnitude and sign (conceptual)
# Also a gentle within-type drift (so posts aren't identical)
drift = np.linspace(-0.15, 0.15, len(df_posts))
support = 1.2*ideo + drift

# Extremity: inversely related to support (your main behavioral result)
extremity = -0.9*support + 0.15*np.sin(np.linspace(0, 3*np.pi, len(df_posts)))

# Polarization salience: higher for more extreme ideology-score magnitude
polar_salience = np.abs(ideo) + 0.08*np.cos(np.linspace(0, 2*np.pi, len(df_posts)))

df_behav = df_posts[["post_id","post_type"]].copy()
df_behav["support_mean"] = support
df_behav["extremity_mean"] = extremity
df_behav["polar_salience"] = polar_salience

df_behav.head()


In [ ]:
def deterministic_group_timeseries(n_sub, n_t, target_corr, phase_shift=0.0):
    """
    Create deterministic time-series with approximate within-group synchrony.
    No randomness: participants share a common signal + structured unique components.
    target_corr controls how dominant the shared signal is.
    """
    t = np.linspace(0, 1, n_t)
    shared = (
        np.sin(2*np.pi*(3*t + phase_shift)) +
        0.35*np.sin(2*np.pi*(9*t + 0.2 + phase_shift)) +
        0.20*np.sin(2*np.pi*(15*t + 0.1))
    )
    shared = (shared - shared.mean()) / shared.std()

    # Convert desired correlation into mixture weight (monotonic mapping)
    # This is conceptual; we just ensure higher target_corr => more shared signal.
    w = np.clip((target_corr - 0.05) / 0.25, 0.05, 0.95)

    X = []
    for i in range(n_sub):
        # structured "individual" component: distinct sinusoids per participant
        indiv = (
            np.sin(2*np.pi*((4+i*0.12)*t + 0.3*i)) +
            0.25*np.sin(2*np.pi*((13+i*0.07)*t + 0.1*i))
        )
        indiv = (indiv - indiv.mean()) / indiv.std()
        x = w*shared + (1-w)*indiv
        X.append(x)
    return np.vstack(X)

def mean_pairwise_corr(X):
    C = np.corrcoef(X)
    iu = np.triu_indices_from(C, k=1)
    return float(np.mean(C[iu]))


### Figure 0: One-post participant vectors across brain regions


In [ ]:
# Select one illustrative post (same for all participants)
example_post = df_posts.query("post_type=='Pro-Left'").iloc[0]

n_participants = 12
n_regions = 96  # conceptual whole-brain parcel set
region_ids = np.arange(1, n_regions + 1)

# Deterministic base spatial pattern tied to the chosen post coordinates
phase = np.linspace(0, 2 * np.pi, n_regions, endpoint=False)
base_pattern = (
    0.9 * np.sin(phase + 0.75 * example_post["x"])
    + 0.6 * np.cos(2 * phase - 0.55 * example_post["y"])
)

# Build 12 participant vectors: shared post pattern + participant-specific modulation
participant_vectors = []
for s in range(n_participants):
    shift = 0.20 * s
    mod = 0.30 * np.sin(3 * phase + shift) + 0.12 * np.cos(5 * phase - 0.15 * s)
    vec = base_pattern + mod
    participant_vectors.append(vec)
participant_vectors = np.vstack(participant_vectors)

# Normalize each participant vector for comparable color scaling
participant_vectors = (
    participant_vectors - participant_vectors.mean(axis=1, keepdims=True)
) / (participant_vectors.std(axis=1, keepdims=True) + 1e-8)

fig, ax = plt.subplots(figsize=(12.5, 4.6), constrained_layout=True)
im = ax.imshow(
    participant_vectors,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-2.2,
    vmax=2.2,
    interpolation="nearest",
)

ax.set_title(
    f"Figure 0. Brain activity vectors for one post across 12 participants\n"
    f"Post ID {int(example_post['post_id'])} ({example_post['post_type']})",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Brain region index")
ax.set_ylabel("Participant")
ax.set_yticks(np.arange(n_participants))
ax.set_yticklabels([f"P{i+1}" for i in range(n_participants)])
ax.set_xticks(np.arange(0, n_regions, 12))
ax.set_xticklabels(np.arange(1, n_regions + 1, 12))

cb = fig.colorbar(im, ax=ax, shrink=0.95)
cb.set_label("Activity (z)")

plt.show()

save_fig(fig, "Figure0_OnePost_12ParticipantVectors")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# ------------------------
# PARAMETERS
# ------------------------
N_POSTS = 12
N_SEGMENTS = 120           # MORE BINS
SEGMENT_WIDTH = 0.25       # MUCH SMALLER WIDTH
STRIP_HEIGHT = 0.5
VERTICAL_SPACING = 0.9

OUTPUT_SVG = "post_vectors_professional_dense.svg"
OUTPUT_PNG = "post_vectors_professional_dense.png"

# Professional muted palette (ordered perceptually)
PALETTE = [
    "#2c7bb6",
    "#00a6ca",
    "#00ccbc",
    "#90eb9d",
    "#ffff8c",
    "#f9d057",
    "#f29e2e",
    "#e76818",
    "#d7191c"
]

# ------------------------
# BASE MOTIFS
# ------------------------

x = np.linspace(0, 2*np.pi, N_SEGMENTS)

motif_A = np.sin(2*x) + 0.4*np.sin(6*x)
motif_B = np.cos(3*x) + 0.5*np.sin(9*x)
motif_C = np.sin(4*x) + 0.3*np.cos(8*x)

def normalize(v):
    return (v - v.min()) / (v.max() - v.min())

motifs = [normalize(m) for m in [motif_A, motif_B, motif_C]]

# ------------------------
# VARIATION
# ------------------------

def make_post_vector(post_idx):
    group = post_idx // 4
    base = motifs[group].copy()

    shift = (post_idx % 4) * 7  # stronger separation between posts
    base = np.roll(base, shift)

    modulation = 1 + 0.04*(post_idx % 4)
    base = base * modulation

    return normalize(base)

# ------------------------
# PLOTTING
# ------------------------

total_width = N_SEGMENTS * SEGMENT_WIDTH

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_xlim(0, total_width)
ax.set_ylim(0, N_POSTS * VERTICAL_SPACING)
ax.axis("off")

for i in range(N_POSTS):
    vec = make_post_vector(i)
    y_base = (N_POSTS - i - 1) * VERTICAL_SPACING

    bins = np.linspace(0, 1, len(PALETTE)+1)
    indices = np.digitize(vec, bins) - 1
    indices = np.clip(indices, 0, len(PALETTE)-1)

    for j in range(N_SEGMENTS):
        rect = Rectangle(
            (j * SEGMENT_WIDTH, y_base),
            SEGMENT_WIDTH,
            STRIP_HEIGHT,
            facecolor=PALETTE[indices[j]],
            edgecolor="none"
        )
        ax.add_patch(rect)

    # Clean outer border
    border = Rectangle(
        (0, y_base),
        total_width,
        STRIP_HEIGHT,
        fill=False,
        edgecolor="black",
        linewidth=0.7
    )
    ax.add_patch(border)

    ax.text(
        -1.2,
        y_base + STRIP_HEIGHT / 2,
        f"Post {i+1:02d}",
        va="center",
        ha="right",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(OUTPUT_SVG, format="svg", bbox_inches="tight")
plt.savefig(OUTPUT_PNG, dpi=300, bbox_inches="tight")

plt.show()

print("Saved:", OUTPUT_SVG)
print("Saved:", OUTPUT_PNG)


### Figure 1: Inter-subject correlation (ISC)

In [ ]:
# Choose a single illustrative condition for time-series (you can swap)
example_left_pt = "Pro-Left"
example_right_pt = "Pro-Right"

left_target = float(df_isc.query("group=='Left' and post_type==@example_left_pt")["mean_isc"])
right_target = float(df_isc.query("group=='Right' and post_type==@example_right_pt")["mean_isc"])

n_sub, n_t = 18, 180
X_left = deterministic_group_timeseries(n_sub, n_t, target_corr=left_target, phase_shift=0.00)
X_right = deterministic_group_timeseries(n_sub, n_t, target_corr=right_target, phase_shift=0.12)

fig = plt.figure(figsize=(13.2, 4.6), constrained_layout=True)
gs = GridSpec(1, 3, figure=fig, width_ratios=[1.25, 1.9, 1.35], wspace=0.35)

# a) Task design
axA = fig.add_subplot(gs[0, 0])
axA.set_title("a  Task design", loc="left", fontweight="bold")
axA.axis("off")
axA.set_xlim(0, 4)
axA.set_ylim(0, 1)

for i, label in enumerate(POST_TYPES):
    r = patches.Rectangle((i + 0.06, 0.52), 0.88, 0.28, facecolor="white", edgecolor="black", lw=0.9)
    axA.add_patch(r)
    axA.text(i + 0.50, 0.66, label, ha="center", va="center")

axA.scatter([0.15], [0.25], color=LEFT_COLOR, s=45)
axA.text(0.28, 0.25, "Left group", va="center")
axA.scatter([0.15], [0.12], color=RIGHT_COLOR, s=45)
axA.text(0.28, 0.12, "Right group", va="center")
axA.text(
    0.06,
    0.02,
    "ISC computed within-group\nseparately for each post type.",
    ha="left",
    va="bottom",
)

# b) Time series synchronization
axB = fig.add_subplot(gs[0, 1])
axB.set_title("b  Within-group time-series synchronization", loc="left", fontweight="bold")
t = np.arange(n_t)

for i in range(6):
    axB.plot(t, X_left[i] + 3.0, color=LEFT_COLOR, alpha=0.25, lw=1.0)
    axB.plot(t, X_right[i] - 3.0, color=RIGHT_COLOR, alpha=0.25, lw=1.0)

axB.plot(t, X_left.mean(0) + 3.0, color=LEFT_COLOR, lw=2.2, label=f"Left ({example_left_pt})")
axB.plot(t, X_right.mean(0) - 3.0, color=RIGHT_COLOR, lw=2.2, label=f"Right ({example_right_pt})")

axB.set_xlabel("Time (TRs / a.u.)")
axB.set_ylabel("BOLD (z)")
axB.legend(frameon=False, loc="upper right")
axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)
axB.text(0.01, 0.04, "ISC = mean pairwise r(time series)", transform=axB.transAxes)

# c) Summary (stacked brain panel + bars to avoid overlap)
right_gs = gs[0, 2].subgridspec(2, 1, height_ratios=[1.0, 1.6], hspace=0.3)
axC_top = fig.add_subplot(right_gs[0, 0])
axC_top.set_title("c  Nilearn template + ISC map", loc="left", fontweight="bold")
sm = draw_brain_like(axC_top, overlay_strength=1.0, mode="isc")
divider_top = make_axes_locatable(axC_top)
cbax_top = divider_top.append_axes("right", size="6%", pad=0.04)
cb_top = fig.colorbar(sm, cax=cbax_top)
cb_top.set_label("r / z", rotation=90)

axC = fig.add_subplot(right_gs[1, 0])
axC.set_title("Mean ISC by post type", loc="left", fontsize=11)
x = np.arange(len(POST_TYPES))
w = 0.38
left_vals = df_isc.query("group=='Left'")["mean_isc"].to_numpy()
right_vals = df_isc.query("group=='Right'")["mean_isc"].to_numpy()

axC.bar(x - w / 2, left_vals, width=w, color=LEFT_COLOR, label="Left")
axC.bar(x + w / 2, right_vals, width=w, color=RIGHT_COLOR, label="Right")
axC.set_xticks(x)
axC.set_xticklabels(POST_TYPES, rotation=20, ha="right")
axC.set_ylabel("ISC (r)")
axC.set_ylim(0, 0.28)
axC.legend(frameon=False, loc="upper right")
axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)

fig.suptitle("Figure 1. Time-domain intersubject correlation (conceptual)", y=1.03, fontweight="bold")
plt.show()

save_fig(fig, "Figure1_ISC_concept")



### Figure 2: Pattern similarity analysis

In [ ]:
def rsm_from_coords(df_posts, group="Left"):
    """
    Deterministic similarity matrix based on 2D semantic coords + group tuning.
    Produces clean block structure by post type, plus slight group-specific emphasis.
    """
    xy = df_posts[["x","y"]].to_numpy()
    # Euclidean distance
    D = np.sqrt(((xy[:,None,:]-xy[None,:,:])**2).sum(-1))
    # Convert to similarity (Gaussian kernel)
    S = np.exp(-(D**2)/(2*(0.55**2)))

    # strengthen within-type similarity
    pt = df_posts["post_type"].to_numpy()
    same_type = (pt[:,None] == pt[None,:]).astype(float)
    S = 0.55*S + 0.45*same_type

    # group-specific "ingroup" sharpening (conceptual)
    if group == "Left":
        ingroup_types = np.isin(pt, ["Pro-Left", "Anti-Right"]).astype(float)
    else:
        ingroup_types = np.isin(pt, ["Pro-Right", "Anti-Left"]).astype(float)

    ingroup_block = np.outer(ingroup_types, ingroup_types)
    S = np.clip(S + 0.10*ingroup_block, 0, 1)

    # force diagonal = 1
    np.fill_diagonal(S, 1.0)

    # map [0..1] to correlation-like [-0.2..0.9] for typical RSM look
    R = -0.2 + 1.1*S
    return R

R_left  = rsm_from_coords(df_posts, "Left")
R_right = rsm_from_coords(df_posts, "Right")


In [ ]:
fig = plt.figure(figsize=(13.2, 4.6), constrained_layout=True)
gs = GridSpec(1, 4, figure=fig, width_ratios=[1.2, 1.15, 1.15, 0.95], wspace=0.4)

# a) Post → pattern schematic
axA = fig.add_subplot(gs[0, 0])
axA.set_title("a  Post-level pattern extraction", loc="left", fontweight="bold")
axA.axis("off")

for i in range(3):
    y = 0.78 - i * 0.28
    post = patches.FancyBboxPatch(
        (0.05, y),
        0.35,
        0.16,
        boxstyle="round,pad=0.02,rounding_size=0.02",
        facecolor="white",
        edgecolor="black",
        lw=0.9,
    )
    axA.add_patch(post)
    axA.text(0.225, y + 0.08, f"Post {i + 1}", ha="center", va="center")
    axA.annotate("", xy=(0.54, y + 0.08), xytext=(0.40, y + 0.08), arrowprops=dict(arrowstyle="-|>", lw=1.0))

    gx0, gy0 = 0.58, y + 0.03
    for r in range(4):
        for c in range(6):
            val = 0.92 - 0.15 * np.cos((c + r + i) * 0.9)
            val = float(np.clip(val, 0.0, 1.0))
            rect = patches.Rectangle(
                (gx0 + c * 0.05, gy0 + r * 0.035),
                0.045,
                0.03,
                facecolor=(val, val, val),
                edgecolor="#d0d0d0",
                lw=0.25,
            )
            axA.add_patch(rect)

axA.text(
    0.05,
    0.03,
    "For each post: average BOLD within post window\n→ spatial pattern per post (per participant).",
    ha="left",
    va="bottom",
)

# b) Left RSM
axB = fig.add_subplot(gs[0, 1])
axB.set_title("b  Left group RSM", loc="left", fontweight="bold")
im1 = axB.imshow(R_left, cmap="RdBu_r", vmin=-0.2, vmax=0.9, origin="lower")
axB.set_xticks([])
axB.set_yticks([])
axB.set_xlabel("Posts")
axB.set_ylabel("Posts")
axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

# c) Right RSM
axC = fig.add_subplot(gs[0, 2])
axC.set_title("c  Right group RSM", loc="left", fontweight="bold")
im2 = axC.imshow(R_right, cmap="RdBu_r", vmin=-0.2, vmax=0.9, origin="lower")
axC.set_xticks([])
axC.set_yticks([])
axC.set_xlabel("Posts")
axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)

# shared colorbar attached to panel c (prevents overlap with layout)
divider_c = make_axes_locatable(axC)
cax = divider_c.append_axes("right", size="6%", pad=0.04)
cb = fig.colorbar(im2, cax=cax)
cb.set_label("Pattern similarity (r)")

# d) Searchlight overlay (Nilearn)
axD = fig.add_subplot(gs[0, 3])
axD.set_title("d  Searchlight: post patterns", loc="left", fontweight="bold")
sm = draw_brain_like(axD, overlay_strength=1.0, mode="pattern")
divider_d = make_axes_locatable(axD)
cax2 = divider_d.append_axes("right", size="6%", pad=0.04)
cb2 = fig.colorbar(sm, cax=cax2)
cb2.set_label("z / q", rotation=90)

fig.suptitle("Figure 2. Post-level pattern similarity (conceptual)", y=1.03, fontweight="bold")
plt.show()

save_fig(fig, "Figure2_PatternSimilarity_concept")



In [ ]:
from nilearn import plotting, image
import matplotlib.pyplot as plt
import numpy as np

isc_path = "/path/to/data/derivatives/isc/parcelwise/left_subjects/parcelwise_isc_AntiLeft_space-MNI152_Schaefer400.nii.gz"   # your NIfTI
isc_img = image.load_img(isc_path)

# Optional: clip extreme values to make plots look cleaner
data = isc_img.get_fdata()
vmax = np.nanpercentile(data, 99)  # robust upper bound
vmin = np.nanpercentile(data, 70)  # lift background a bit (tune)

fig = plt.figure(figsize=(8, 3.2))
display = plotting.plot_stat_map(
    isc_img,
    display_mode="ortho",
    cut_coords=(0, -10, 20),        # tweak for your clusters
    threshold=vmin,                 # key: removes haze
    vmax=vmax,
    cmap="magma",                   # nice for positive maps; try "viridis" too
    colorbar=True,
    annotate=False,
    draw_cross=False,
    black_bg=False,
    title="Within-group ISC (Fisher z)"
)

plt.savefig("isc_statmap.png", dpi=300, bbox_inches="tight")
plt.savefig("isc_statmap.svg", bbox_inches="tight")
plt.show()


In [ ]:
from nilearn import plotting, image
import numpy as np
import matplotlib.pyplot as plt

isc_path = "/path/to/data/derivatives/isc/parcelwise/left_subjects/parcelwise_isc_AntiLeft_space-MNI152_Schaefer400.nii.gz"   # your NIfTI
isc_img = image.load_img(isc_path)

data = isc_img.get_fdata()
vmax = np.nanpercentile(data, 99)
thr  = np.nanpercentile(data, 90)

fig = plt.figure(figsize=(7, 2.8))
plotting.plot_glass_brain(
    isc_img,
    display_mode="lyrz",
    threshold=thr,
    vmax=vmax,
    cmap="plasma",
    colorbar=True,
    plot_abs=False,
    title="ISC (glass brain)"
)
plt.savefig("isc_glass.png", dpi=300, bbox_inches="tight")
plt.savefig("isc_glass.svg", bbox_inches="tight")
plt.show()


### Figure 3: Representational similarity analysis (RSA)

In [ ]:
def rdm_from_similarity(S):
    return 1.0 - S

# Use R_left as "neural similarity" proxy (conceptual). Convert to [0..1] similarity first.
# Reverse the earlier mapping: R = -0.2 + 1.1*S  =>  S = (R + 0.2) / 1.1
S_neural = np.clip((R_left + 0.2)/1.1, 0, 1)  # pretend this is neural similarity
neural_rdm = rdm_from_similarity(S_neural)

# Behavioral RDM from df_behav features (support, extremity, polar_salience)
B = df_behav[["support_mean","extremity_mean","polar_salience"]].to_numpy()
# standardize columns (deterministic)
B = (B - B.mean(axis=0, keepdims=True)) / (B.std(axis=0, keepdims=True) + 1e-8)

diff = B[:,None,:] - B[None,:,:]
behav_rdm = np.sqrt((diff**2).sum(-1))

# RSA (Spearman-like using rank transform without scipy)
iu = np.triu_indices_from(neural_rdm, k=1)
x = neural_rdm[iu]
y = behav_rdm[iu]
xr = x.argsort().argsort().astype(float)
yr = y.argsort().argsort().astype(float)
rho = np.corrcoef(xr, yr)[0,1]
rho


In [ ]:
fig = plt.figure(figsize=(13.2, 4.6), constrained_layout=True)
gs = GridSpec(1, 4, figure=fig, width_ratios=[1.05, 1.05, 1.35, 0.95], wspace=0.4)

# a) Neural RDM
axA = fig.add_subplot(gs[0, 0])
axA.set_title("a  Neural RDM", loc="left", fontweight="bold")
im1 = axA.imshow(neural_rdm, cmap="RdBu_r", origin="lower")
axA.set_xticks([])
axA.set_yticks([])
axA.set_xlabel("Posts")
axA.set_ylabel("Posts")
axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)

# b) Behavioral RDM
axB = fig.add_subplot(gs[0, 1])
axB.set_title("b  Behavioral RDM", loc="left", fontweight="bold")
im2 = axB.imshow(behav_rdm, cmap="RdBu_r", origin="lower")
axB.set_xticks([])
axB.set_yticks([])
axB.set_xlabel("Posts")
axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

# shared colorbar attached to panel b
divider_b = make_axes_locatable(axB)
cax = divider_b.append_axes("right", size="6%", pad=0.04)
cb = fig.colorbar(im2, cax=cax)
cb.set_label("Dissimilarity (a.u.)")

# c) RSA scatter
axC = fig.add_subplot(gs[0, 2])
axC.set_title("c  RSA: neural vs behavioral structure", loc="left", fontweight="bold")

idx = np.linspace(0, len(x) - 1, 2200).astype(int)
axC.scatter(x[idx], y[idx], s=7, alpha=0.22, color="black")

axC.set_xlabel("Neural dissimilarity (1 − similarity)")
axC.set_ylabel("Behavioral distance")
axC.spines["top"].set_visible(False)
axC.spines["right"].set_visible(False)
axC.text(0.03, 0.97, f"Spearman ρ ≈ {rho:.2f}", transform=axC.transAxes, ha="left", va="top", fontweight="bold")
axC.text(
    0.03,
    0.90,
    "Behavioral distance from:\nSupport, Extremity, Polarization",
    transform=axC.transAxes,
    ha="left",
    va="top",
)

# d) Searchlight RSA overlay (Nilearn)
axD = fig.add_subplot(gs[0, 3])
axD.set_title("d  Searchlight RSA", loc="left", fontweight="bold")
sm = draw_brain_like(axD, overlay_strength=1.0, mode="rsa")
divider_d = make_axes_locatable(axD)
cax2 = divider_d.append_axes("right", size="6%", pad=0.04)
cb2 = fig.colorbar(sm, cax=cax2)
cb2.set_label("ρ / z", rotation=90)

fig.suptitle("Figure 3. RSA linking neural and behavioral structure (conceptual)", y=1.03, fontweight="bold")
plt.show()

save_fig(fig, "Figure3_RSA_concept")



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

POST_TYPES = ["Pro-Left", "Anti-Right", "Pro-Right", "Anti-Left"]
N_PER_TYPE = 18
N_POSTS = len(POST_TYPES) * N_PER_TYPE

# Posts dataframe
df_posts = pd.DataFrame({
    "post_id": np.arange(N_POSTS),
    "post_type": np.repeat(POST_TYPES, N_PER_TYPE)
})

# Deterministic 2D "neural representational space" coordinates for posts
centers = {
    "Pro-Left":   np.array([+2.0, +0.7]),
    "Anti-Right": np.array([+1.3, -0.7]),
    "Pro-Right":  np.array([-2.0, -0.7]),
    "Anti-Left":  np.array([-1.3, +0.7]),
}
angles = np.linspace(0, 2*np.pi, N_PER_TYPE, endpoint=False)
offsets = np.c_[0.30*np.cos(angles), 0.22*np.sin(angles)]

coords = np.vstack([centers[pt] + offsets for pt in POST_TYPES])
df_posts["x"] = coords[:, 0]
df_posts["y"] = coords[:, 1]

# Deterministic behavioral features per post (post means)
# Encode your finding: higher support -> lower perceived extremity (inverse relation)
type_ideo = {"Pro-Left": +1.0, "Anti-Right": +0.7, "Pro-Right": -1.0, "Anti-Left": -0.7}
ideo = df_posts["post_type"].map(type_ideo).to_numpy()

drift = np.linspace(-0.15, 0.15, N_POSTS)  # within-type smooth variation
support = 1.2 * ideo + drift
extremity = -0.9 * support + 0.12 * np.sin(np.linspace(0, 3*np.pi, N_POSTS))
polar_salience = np.abs(ideo) + 0.08 * np.cos(np.linspace(0, 2*np.pi, N_POSTS))

df_behav = df_posts[["post_id","post_type"]].copy()
df_behav["support_mean"] = support
df_behav["extremity_mean"] = extremity
df_behav["polar_salience"] = polar_salience

df_posts.head(), df_behav.head()


In [ ]:
def gaussian_similarity_from_coords(xy, sigma=0.85):
    D = np.sqrt(((xy[:, None, :] - xy[None, :, :])**2).sum(-1))
    S = np.exp(-(D**2) / (2*sigma**2))
    np.fill_diagonal(S, 1.0)
    return S

# --- Neural similarity (conceptual) ---
xy = df_posts[["x","y"]].to_numpy()
S_neural = gaussian_similarity_from_coords(xy, sigma=0.85)

# Boost within-post-type similarity to get clear block structure
pt = df_posts["post_type"].to_numpy()
same_type = (pt[:, None] == pt[None, :]).astype(float)
S_neural = 0.60*S_neural + 0.40*same_type
S_neural = np.clip(S_neural, 0, 1)
np.fill_diagonal(S_neural, 1.0)

RDM_neural = 1 - S_neural  # dissimilarity

# --- Behavioral RDM from ratings ---
B = df_behav[["support_mean","extremity_mean","polar_salience"]].to_numpy()
B = (B - B.mean(axis=0, keepdims=True)) / (B.std(axis=0, keepdims=True) + 1e-8)

diff = B[:, None, :] - B[None, :, :]
RDM_behav = np.sqrt((diff**2).sum(-1))  # Euclidean distance

RDM_neural.shape, RDM_behav.shape


In [ ]:
def upper_tri_vec(M):
    iu = np.triu_indices_from(M, k=1)
    return M[iu]

x = upper_tri_vec(RDM_neural)
y = upper_tri_vec(RDM_behav)

rho, pval = spearmanr(x, y)

# Optional deterministic permutation test:
# cycle-shift post labels to generate null distribution
def perm_test_cycle_shift(rdm_a, rdm_b, n_perm=200):
    base = upper_tri_vec(rdm_a)
    null = []
    n = rdm_b.shape[0]
    for k in range(1, n_perm+1):
        idx = np.roll(np.arange(n), k)  # deterministic shift
        perm = rdm_b[idx][:, idx]
        r, _ = spearmanr(base, upper_tri_vec(perm))
        null.append(r)
    null = np.array(null)
    p = (np.sum(np.abs(null) >= np.abs(rho)) + 1) / (len(null) + 1)
    return null, p

null_rhos, p_perm = perm_test_cycle_shift(RDM_neural, RDM_behav, n_perm=200)

rho, pval, p_perm


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4), constrained_layout=True)

# a) Neural RDM
ax = axes[0]
im1 = ax.imshow(RDM_neural, cmap="viridis", origin="lower", vmin=0, vmax=1)
ax.set_title("Neural RDM (response to posts)", loc="left", fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])
cb1 = fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.02)
cb1.set_label("Dissimilarity")

# b) Behavioral RDM
ax = axes[1]
im2 = ax.imshow(RDM_behav, cmap="viridis", origin="lower")
ax.set_title("Behavioral RDM (post ratings)", loc="left", fontweight="bold")
ax.set_xticks([]); ax.set_yticks([])
cb2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.02)
cb2.set_label("Distance")

# c) RSA scatter
ax = axes[2]
idx = np.linspace(0, len(x)-1, 2500).astype(int)  # deterministic downsample for clarity
ax.scatter(x[idx], y[idx], s=8, alpha=0.22, color="black")
ax.set_title("RSA", loc="left", fontweight="bold")
ax.set_xlabel("Neural dissimilarity (1 − similarity)")
ax.set_ylabel("Behavioral distance")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

fig.savefig("RSA_post_level_panels.svg", bbox_inches="tight")
fig.savefig("RSA_post_level_panels.png", dpi=300, bbox_inches="tight")
print("Saved RSA_post_level_panels.svg / RSA_post_level_panels.png")
